# Week 6 — Image Restoration


Complete the `TODO` cells. Do not modify the validation cells.

## Degradation model and derivation

In the spatial domain a linear, position-invariant degradation with additive noise is

$$g(x) = h(x) \circledast f(x) + \eta(x) = \sum_{m=0}^{M-1} h(m)\, f\big((x-m) \bmod M\big) + \eta(x)$$

The DFT convolution theorem says circular convolution becomes a point-wise product, and
the DFT is linear, so the additive noise stays additive:

$$G(u) = H(u)\,F(u) + N(u)$$

(In 2-D this is exactly the familiar $G(u,v) = H(u,v)F(u,v) + N(u,v)$.)

So computing $G$ point-wise and taking `np.fft.ifft` must reproduce the spatial-domain
result bit-for-bit (up to floating-point round-off). Note the convolution must be
**circular**: the DFT treats the signal as periodic, so `np.convolve` (linear convolution,
zero padding) would *not* match.

In [2]:
import numpy as np

f = np.array([1, 2, 1, 2], dtype=float)
h = np.array([0.5, 0.5, 0, 0], dtype=float)
eta = np.array([0.2, -0.1, 0.3, 0.0])

M = f.size

# Circular convolution of h and f, straight from the definition:
#   (h (*) f)(x) = sum_m h(m) * f((x - m) mod M)
# The modulo is what makes it wrap around, which is exactly what the DFT assumes.
conv = np.array([np.sum(h * f[(x - np.arange(M)) % M]) for x in range(M)])

# g = h (*) f + eta
g_spatial = conv + eta

# Frequency domain: G(u) = H(u) F(u) + N(u)
F = np.fft.fft(f)
H = np.fft.fft(h)
N_freq = np.fft.fft(eta)
G = H * F + N_freq
g_from_frequency = np.fft.ifft(G)

print("g (spatial):", g_spatial)
print("g (from frequency domain):", np.round(g_from_frequency.real, 6) if g_from_frequency is not None else None)

g (spatial): [1.7 1.4 1.8 1.5]
g (from frequency domain): [1.7 1.4 1.8 1.5]


In [3]:
# Validation -- do not modify

assert g_spatial is not None and g_from_frequency is not None, "Fill in the TODOs above first."
assert g_spatial.shape == (4,)
np.testing.assert_allclose(g_spatial, g_from_frequency.real, atol=1e-10)
print("Derivation verified: your G(u,v) formula produces exactly the same g as direct spatial-domain computation.")

Derivation verified: your G(u,v) formula produces exactly the same g as direct spatial-domain computation.
